<div class="alert alert-info">
    <b>1.2 Datenqualität: Ionenbilanzfehler (Bereinigte Daten - All Ions)</b><br>
    Dieses Notebook analysiert den Ionenbilanzfehler (IBE) basierend auf dem globalen Datenschema.<br>
    -> Nutzung definierter Standard-Ionen mit entsprechenden Spaltennamen (Regex)<br>
    -> Berücksichtigung von H+ über pH-Werte<br>
    -> Erweiterter Bericht mit Unusable-Statistik
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os
import glob
import re
import textwrap

# ------------------------- Konfiguration der Pfade -------------------------
INPUT_DIR = "../../../../../../1_Acquisition/1.1_Data-Acquisition-Wrapper/Angepasste_Datenbanken"
OUTPUT_PDF = "Adjusted_Data_IBE_Report.pdf"

<div class="alert alert-info">
    <b>Ziel-Ionen Definition</b><br>
    -> Regex-Pattern für die Standard-Spaltennamen (z.B. 'Na_in_mg/L')<br>
    -> Hinterlegung von Valenzen für die Umrechnung
</div>

In [2]:
# ------------------------- Ziel-Ionen Konfiguration -------------------------
TARGET_IONS = {
    'Na': {'valence': 1, 'type': 'cation', 'regex': r'Na_in_([a-zA-Z0-9/]+)'},
    'K': {'valence': 1, 'type': 'cation', 'regex': r'K_in_([a-zA-Z0-9/]+)'},
    'Ca': {'valence': 2, 'type': 'cation', 'regex': r'Ca_in_([a-zA-Z0-9/]+)'},
    'Mg': {'valence': 2, 'type': 'cation', 'regex': r'Mg_in_([a-zA-Z0-9/]+)'},
    'Li': {'valence': 1, 'type': 'cation', 'regex': r'Li_in_([a-zA-Z0-9/]+)'},
    'Sr': {'valence': 2, 'type': 'cation', 'regex': r'Sr_in_([a-zA-Z0-9/]+)'},
    'NH4': {'valence': 1, 'type': 'cation', 'regex': r'NH4_in_([a-zA-Z0-9/]+)'},
    'Fe': {'valence': 2, 'type': 'cation', 'regex': r'Fe_in_([a-zA-Z0-9/]+)'},
    'Mn': {'valence': 2, 'type': 'cation', 'regex': r'Mn_in_([a-zA-Z0-9/]+)'},
    
    'Cl': {'valence': 1, 'type': 'anion', 'regex': r'Cl_in_([a-zA-Z0-9/]+)'},
    'HCO3': {'valence': 1, 'type': 'anion', 'regex': r'HCO3_in_([a-zA-Z0-9/]+)'},
    'SO4': {'valence': 2, 'type': 'anion', 'regex': r'SO4_in_([a-zA-Z0-9/]+)'},
    'NO3': {'valence': 1, 'type': 'anion', 'regex': r'NO3_in_([a-zA-Z0-9/]+)'},
    'F': {'valence': 1, 'type': 'anion', 'regex': r'F_in_([a-zA-Z0-9/]+)'},
    'HS': {'valence': 1, 'type': 'anion', 'regex': r'HS_in_([a-zA-Z0-9/]+)'},
}

<div class="alert alert-info">
    <b>IBE-Berechnung</b><br>
    -> Hilfsfunktion zur Faktor-Berechnung<br>
    -> Dynamisches Finden der Spalten basierend auf Regex<br>
    -> Hinzufügen von H+ (falls pH vorhanden)
</div>

In [3]:
def get_conversion_factor(unit, valence):
    # ------------------------- Gibt den Umrechnungsfaktor zu meq/L basierend auf dem Einheits-String zurück -------------------------
    unit = unit.lower()
    if unit == 'mmol/l':
        return valence
    elif unit == 'umol/l' or unit == '\u00b5mol/l':
        return valence / 1000.0
    elif unit == 'meq/l':
        return 1.0
    return 0

def calculate_ibe(df):
    # ------------------------- IBE durch parsen -------------------------
    cations_sum = 0
    anions_sum = 0
    found_ions_list = []
    
    # ------------------------- Verfügbare Spalten identifizieren -------------------------
    ion_cols = {}
    for ion, info in TARGET_IONS.items():
        for col in df.columns:
            match = re.match(info['regex'], col)
            if match:
                unit = match.group(1)
                ion_cols[ion] = {'col': col, 'unit': unit, 'info': info}
                break
    
    # ------------------------- Summierung -------------------------
    for ion, data in ion_cols.items():
        col = data['col']
        unit = data['unit']
        info = data['info']
        
        factor = get_conversion_factor(unit, info['valence'])
        if factor == 0:
            print(f"Warnung: Einheit '{unit}' nicht unterstützt (Spalte: {col})")
            continue
            
        vals = pd.to_numeric(df[col], errors='coerce').fillna(0)
        meq_vals = vals * factor
        
        if info['type'] == 'cation':
            cations_sum += meq_vals
        else:
            anions_sum += meq_vals
        found_ions_list.append(f"{ion}({unit})")
        
    # ------------------------- H+ Integration -------------------------
    ph_col = None
    for col in df.columns:
        if col.strip().lower() == 'ph':
            ph_col = col
            break
    
    if ph_col:
        try:
            ph_vals = pd.to_numeric(df[ph_col], errors='coerce')
            # ------------------------- 10^(-pH) * 1000 = meq/L -------------------------
            h_meq = (10 ** (-ph_vals)) * 1000.0
            h_meq = h_meq.fillna(0)
            cations_sum += h_meq
            found_ions_list.append("H+(pH)")
        except:
            pass
            
    total_ions = cations_sum + anions_sum
    
    # ------------------------- IBE Berechnung -------------------------
    ibe = ((cations_sum - anions_sum) / total_ions) * 100
    
    # ------------------------- Filtere Zeilen mit wenig Ionen -------------------------
    ibe = ibe[total_ions > 0.1]
    
    return ibe, found_ions_list

<div class="alert alert-info">
    <b>Hauptverarbeitung & Bericht</b><br>
    -> Einlesen aller bereinigten .csv Dateien<br>
    -> Erstellung der Grafiken pro Datei<br>
    -> Finale Zusammenfassung über alle Datenbanken
</div>

In [4]:
files = glob.glob(os.path.join(INPUT_DIR, "*cleaned_Database.csv"))
print(f"Gefunden: {len(files)} bereinigte Datenbanken.")

# ------------------------- Globale Variablen -------------------------
global_total_records = 0
global_good_records = 0
db_stats = []

# ------------------------- PDF Bericht -------------------------
with PdfPages(OUTPUT_PDF) as pdf:
    for filepath in files:
        filename = os.path.basename(filepath)
        try:
            df = pd.read_csv(filepath)
            
            ibe, found_ions = calculate_ibe(df)
            
            # ------------------------- Update Stats -------------------------
            n_total = len(ibe)
            n_good = (ibe.abs() <= 5).sum()
            
            global_total_records += n_total
            global_good_records += n_good
            
            db_stats.append({
                'name': filename[:40],
                'total': n_total,
                'good': n_good
            })
            
            # ------------------------- Plot erstellen -------------------------
            plt.figure(figsize=(10, 7))
            if len(ibe) > 0:
                plt.hist(ibe, bins=30, range=(-50, 50), edgecolor='black', alpha=0.7)
                plt.title(f"IBE (Dynamisch + H+): {filename}\n(N={n_total})")
                plt.xlabel("IBE (%)")
                plt.ylabel("Anzahl")
                plt.grid(True, alpha=0.3)
                
                # ------------------------- Textbox -------------------------
                ions_str = ", ".join(found_ions)
                if not ions_str: ions_str = "Keine gefunden"
                wrapped_ions = textwrap.fill(ions_str, width=65)
                
                mean_val = ibe.mean()
                median_val = ibe.median()
                pct_5 = (n_good / n_total * 100) if n_total > 0 else 0
                
                stats_str = (
                    f"Mean: {mean_val:.2f}%\n"
                    f"Median: {median_val:.2f}%\n"
                )
                
                stats_5pct = f"+/- 5%: {pct_5:.1f}% ({n_good}/{n_total})"
                
                ions_block = f"\n\nIons Used:\n{wrapped_ions}"

                plt.text(1.02, 1.0, stats_str, transform=plt.gca().transAxes, 
                         fontsize=10, verticalalignment='top')
                         
                # ------------------------- Highlight rot für 5% -------------------------
                plt.text(1.02, 0.90, stats_5pct, transform=plt.gca().transAxes, 
                         fontsize=11, fontweight='bold', color='red', verticalalignment='top')
                
                plt.text(1.02, 0.85, ions_block, transform=plt.gca().transAxes, 
                         fontsize=8, verticalalignment='top')
                
                plt.tight_layout()
            else:
                plt.text(0.5, 0.5, "Unzureichende Daten", 
                         horizontalalignment='center', verticalalignment='center')
                plt.title(f"IBE: {filename}")
                
            pdf.savefig()
            plt.close()
            print(f"Verarbeitet: {filename} ({len(ibe)} Records)")
        except Exception as e:
            print(f"Fehler {filename}: {e}")
            db_stats.append({'name': filename[:30], 'total': 0, 'good': 0})
            
    # ------------------------- Zusammenfassung ------------------------- 
    items_per_page = 35 
    total_dbs = len(db_stats)
    
    current_idx = 0
    page_num = 1
    
    print(f"Erstelle Zusammenfassung...")
    
    while current_idx < total_dbs:
        plt.figure(figsize=(11.69, 8.27)) 
        
        # ------------------------- Fixierte Koordinaten -------------------------
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.axis('off') 
        
        end_idx = min(current_idx + items_per_page, total_dbs)
        
        # ------------------------- Header -------------------------
        plt.text(0.5, 0.97, f"Global Data Quality Summary (Page {page_num})", 
                 fontsize=16, fontweight='bold', ha='center', va='top')
        
        y_pos = 0.90
        
        if page_num == 1:
            if global_total_records > 0:
                global_pct = (global_good_records / global_total_records) * 100
            else:
                 global_pct = 0
            
            g_text = (f"OVERALL: {global_good_records}/{global_total_records} records good ({global_pct:.1f}%)")
            plt.text(0.5, y_pos, g_text, fontsize=12, fontweight='bold', ha='center', color='blue')
            y_pos -= 0.05
        
        header_str = f"{'Database':<55} | {'Total':<10} | {'+/- 5% (Good)':<15} | {'Unusable':<10}"
        plt.text(0.05, y_pos, header_str, fontsize=10, fontweight='bold', fontfamily='monospace')
        
        plt.plot([0.05, 0.95], [y_pos-0.01, y_pos-0.01], color='black', linewidth=1)
        
        y_pos -= 0.035
        
        for i in range(current_idx, end_idx):
            stat = db_stats[i]
            rem = stat['total'] - stat['good']
            name_disp = (stat['name'][:53] + '..') if len(stat['name']) > 53 else stat['name']
            
            line = f"{name_disp:<55} | {stat['total']:<10} | {stat['good']:<15} | {rem:<10}"
            
            plt.text(0.05, y_pos, line, fontsize=9, fontfamily='monospace')
            y_pos -= 0.02
            
        pdf.savefig()
        plt.close()
        
        current_idx = end_idx
        page_num += 1

print(f"\nBericht erstellt: {OUTPUT_PDF}")

Gefunden: 0 bereinigte Datenbanken.
Erstelle Zusammenfassung...

Bericht erstellt: Adjusted_Data_IBE_Report.pdf


C:\Users\lucca\AppData\Local\Temp\ipykernel_31140\2354411206.py:10: MatplotlibDeprecationWarning: Keeping empty pdf files is deprecated since 3.8 and support will be removed in 3.10.
  with PdfPages(OUTPUT_PDF) as pdf:
